# 🦀 Day 7: Core Logic — Part 2: Persistence and Protocol
---

Today we add persistence and a simple text protocol to RustKV.

- **Write-Ahead Log (WAL)**: Append-only file for durability
- **Persistence**: Save/load snapshots, serialization format
- **Protocol**: Simple text protocol (Redis RESP-like)
- **Command parsing**: `"SET key value"` → `Command::Set(...)`
- **Response formatting**: `"+OK\r\n"`, `"$5\r\nvalue\r\n"`

## Write-Ahead Log (WAL) Concept

1. Every write (SET, DELETE) is appended to a log file
2. On crash, replay the log to recover state
3. Periodic snapshots reduce log size (optional)

**Simple format**: One command per line, e.g. `SET key value` or `DEL key`

In [ ]:
use std::fs::OpenOptions;
use std::io::{BufRead, BufReader, Write};

/// Simple WAL writer — append commands to file
pub struct WalWriter {
    file: std::fs::File,
}

impl WalWriter {
    pub fn new(path: &str) -> std::io::Result<Self> {
        let file = OpenOptions::new()
            .create(true)
            .append(true)
            .open(path)?;
        Ok(Self { file })
    }

    pub fn write_set(&mut self, key: &str, value: &str) -> std::io::Result<()> {
        writeln!(self.file, "SET {} {}", key, value)
    }

    pub fn write_del(&mut self, key: &str) -> std::io::Result<()> {
        writeln!(self.file, "DEL {}", key)
    }

    pub fn flush(&mut self) -> std::io::Result<()> {
        self.file.flush()
    }
}

/// Simple WAL reader — replay log
pub fn replay_wal(path: &str) -> std::io::Result<Vec<(String, String)>> {
    let file = std::fs::File::open(path)?;
    let reader = BufReader::new(file);
    let mut data: Vec<(String, String)> = Vec::new();
    for line in reader.lines() {
        let line = line?;
        let parts: Vec<&str> = line.splitn(3, ' ').collect();
        if parts.len() >= 3 && parts[0] == "SET" {
            data.push((parts[1].to_string(), parts[2].to_string()));
        }
    }
    Ok(data)
}

println!("WAL writer and reader defined!");

In [ ]:
// Protocol: simple text format (RESP-like)

#[derive(Debug, Clone, PartialEq)]
pub enum Command {
    Get(String),
    Set(String, String),
    Delete(String),
    Keys(String),
    Ping,
    Quit,
}

#[derive(Debug, Clone, PartialEq)]
pub enum Response {
    Value(String),
    Ok,
    Error(String),
    Nil,
    Array(Vec<String>),
}

/// Parse "SET key value" -> Command::Set(key, value)
pub fn parse_command(line: &str) -> Option<Command> {
    let parts: Vec<&str> = line.trim().splitn(3, ' ').collect();
    match parts.first()?.to_uppercase().as_str() {
        "GET" if parts.len() >= 2 => Some(Command::Get(parts[1].to_string())),
        "SET" if parts.len() >= 3 => Some(Command::Set(parts[1].to_string(), parts[2].to_string())),
        "DEL" | "DELETE" if parts.len() >= 2 => Some(Command::Delete(parts[1].to_string())),
        "KEYS" if parts.len() >= 2 => Some(Command::Keys(parts[1].to_string())),
        "PING" => Some(Command::Ping),
        "QUIT" => Some(Command::Quit),
        _ => None,
    }
}

/// Format response as RESP-like: +OK\r\n, $5\r\nvalue\r\n
pub fn format_response(resp: &Response) -> String {
    match resp {
        Response::Ok => "+OK\r\n".to_string(),
        Response::Nil => "$-1\r\n".to_string(),
        Response::Value(s) => format!("${}\r\n{}\r\n", s.len(), s),
        Response::Error(e) => format!("-{}\r\n", e),
        Response::Array(arr) => {
            let mut out = format!("*{}\r\n", arr.len());
            for s in arr {
                out.push_str(&format!("${}\r\n{}\r\n", s.len(), s));
            }
            out
        }
    }
}

let cmd = parse_command("SET name RustKV").unwrap();
assert_eq!(cmd, Command::Set("name".into(), "RustKV".into()));
let resp = format_response(&Response::Ok);
assert_eq!(resp, "+OK\r\n");
println!("Parse: {:?}", cmd);
println!("Format: {:?}", resp);

In [ ]:
// Test persistence round-trip (in-memory, no actual file for notebook)

let commands = [
    "SET a 1",
    "SET b 2",
    "DEL a",
];

let mut parsed: Vec<Command> = Vec::new();
for line in commands {
    if let Some(c) = parse_command(line) {
        parsed.push(c);
    }
}

assert_eq!(parsed.len(), 3);
assert_eq!(parsed[0], Command::Set("a".into(), "1".into()));
assert_eq!(parsed[1], Command::Set("b".into(), "2".into()));
assert_eq!(parsed[2], Command::Delete("a".into()));

// Response formatting
assert_eq!(format_response(&Response::Value("hello".into())), "$5\r\nhello\r\n");
assert_eq!(format_response(&Response::Array(vec!["a".into(), "b".into()])), "*2\r\n$1\r\na\r\n$1\r\nb\r\n");

println!("Persistence and protocol tests passed!");

## 📝 Exercise: Add Support for New Commands

Add support for:
- **EXISTS key** — returns 1 if key exists, 0 otherwise
- **INCR key** — increment integer value by 1, return new value

1. Add `Exists(key)` and `Incr(key)` to `Command` enum
2. Update `parse_command` to handle `EXISTS` and `INCR`
3. Add `Integer(i64)` to `Response` if needed
4. Update `format_response` for integer responses (RESP: `:42\r\n`)

In [ ]:
// YOUR CODE HERE
// Add EXISTS and INCR commands

// Extend Command:
// pub enum Command {
//     ...
//     Exists(String),
//     Incr(String),
// }

// Extend Response:
// pub enum Response {
//     ...
//     Integer(i64),
// }

// In format_response: Response::Integer(i) => format!(":{}\r\n", i)

// In parse_command: "EXISTS" => Some(Command::Exists(...)), "INCR" => Some(Command::Incr(...))

println!("Add EXISTS and INCR to complete the exercise!");

## 🎯 Key Takeaways

1. **WAL**: Append-only log for durability; replay on startup to recover
2. **Simple format**: One command per line (`SET key value`, `DEL key`)
3. **Protocol**: Text-based, RESP-like (`+OK\r\n`, `$5\r\nvalue\r\n`, `$-1\r\n` for Nil)
4. **Command parsing**: Split by space, match first token, validate args
5. **Response formatting**: Different prefixes for Ok, Value, Error, Nil, Array
6. **Extensibility**: Add new commands by extending `Command` and `parse_command`

---

**Week 21 Complete!** Core engine is built. Next week: features, testing, and polish!